# Notebook 2: Undersampling and aliasing

*Part of MRI Reconstruction from k-space to Deep Learning: A Hands-on Introduction*

In Notebook 1 you saw that measuring every second line of k-space makes the object appear twice. Yet modern scanners routinely measure only a quarter of k-space and still produce clean images. The trick starts with which lines you choose to measure. This notebook is about that choice.

**Learning objectives.** By the end of this notebook you will be able to:

1. Explain why skipping k-space lines makes a scan faster, and define the acceleration factor R.
2. Predict the aliasing produced by regular undersampling: how many copies, and where.
3. Use the point spread function of a sampling pattern to predict its artifacts without any image.
4. Explain why random and variable-density sampling turn copies into noise-like artifacts, and why that makes reconstruction possible.
5. Compare sampling patterns with a simple error measure, NRMSE.

**Prerequisites:** Notebook 1 (k-space, center vs edges, every k-space point affects every pixel).

**Time:** about 1.5–2 hours on your own.

As before, write down your answer to every Predict question before running the cell.

## 1. Setup

The first cell imports the libraries. The second repeats the helper functions from Notebook 1, adds two small new ones, and creates the phantom and its full k-space, which we treat as the ground truth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order

In [ ]:
# From Notebook 1
def fft2c(image):
    """Centered, orthonormal 2D FFT: image -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image), norm="ortho"))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT: k-space -> image."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace), norm="ortho"))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        if im.dtype == bool:  # sampling masks: white = measured, black = skipped, with a thin frame
            ax.imshow(im, cmap=cmap, vmin=0, vmax=1, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


# New in this notebook
def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


# The object and its fully sampled k-space
N = 256
image = resize(shepp_logan_phantom(), (N, N), anti_aliasing=True)
kspace = fft2c(image)
print(f"Ground truth: {N} x {N} phantom; full k-space has {N} phase-encode lines")

## 2. Why we skip lines, not points

A scanner fills k-space one line at a time. In each repetition, it applies a *phase-encoding* gradient that picks one row $k_y$, then records the whole row along $k_x$ (the *readout*) in a few milliseconds. The next repetition picks another row. So:

$$ \text{scan time} \approx \text{number of lines measured} \times \text{repetition time (TR)} $$

Skipping points *within* a line saves no time, because the whole line is recorded anyway. Skipping whole lines does. Measuring only $N/R$ of the $N$ lines makes the scan R times faster; $R$ is called the acceleration factor.

All sampling patterns in this notebook are therefore a choice of which rows of k-space to keep. The helper functions below create such patterns:

- `regular_lines`: every R-th line (what you tried at the end of Notebook 1).
- `random_lines`: lines picked at random, optionally with a fully sampled block in the center.
- `variable_density_lines`: random lines, but much more likely near the center.
- `zero_filled`: the simplest reconstruction. Unmeasured lines are set to zero, then the inverse FFT is applied.

In [ ]:
def regular_lines(N, R):
    """Every R-th phase-encode line, always including the center line."""
    lines = np.zeros(N, dtype=bool)
    lines[(N // 2) % R :: R] = True
    return lines


def random_lines(N, R, center_fraction=0.0, seed=0):
    """N // R lines: a fully sampled central block, the rest chosen uniformly at random."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    lines[rng.choice(candidates, size=n_total - n_center, replace=False)] = True
    return lines


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest chosen at random,
    with lines near the center much more likely to be picked than lines near the edge."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)       # 0 at the center, 1 at the edge
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


def as_mask(lines):
    """Turn a 1D set of phase-encode lines into a 2D k-space mask (each line covers every column)."""
    return np.repeat(lines[:, None], lines.size, axis=1)


def zero_filled(kspace, lines):
    """Simplest reconstruction: treat unmeasured k-space as zero and apply the inverse FFT."""
    return np.abs(ifft2c(kspace * lines[:, None]))

In [ ]:
full = np.ones(N, dtype=bool)
quarter = regular_lines(N, 4)

print(f"Full scan: {full.sum()} lines.  R = 4: {quarter.sum()} lines, so 4x faster.")
show([as_mask(full), as_mask(quarter)], ["fully sampled k-space", "R = 4: every 4th line"])

White rows are measured, black rows are skipped. Each row in the mask is one repetition of the scanner.

## 3. Regular undersampling: coherent aliasing

> **Predict:** At the end of Notebook 1, keeping every 2nd line gave 2 overlapping copies. What will you see when you keep every 3rd or every 4th line? How many copies, and how far apart?

In [ ]:
Rs = [2, 3, 4]
masks = [regular_lines(N, R) for R in Rs]

ordinal = {2: "2nd", 3: "3rd", 4: "4th"}
show([as_mask(m) for m in masks], [f"every {ordinal[R]} line (R = {R})" for R in Rs])
show([zero_filled(kspace, m) for m in masks], [f"zero-filled, R = {R}" for R in Rs], vmin=0, vmax=1)

Keeping every R-th line produces R copies of the object, shifted by $N/R$ pixels from each other, each with $1/R$ of the brightness. The field of view in the vertical direction shrinks to $N/R$, and whatever does not fit wraps around.

This is called coherent aliasing: the artifact is a sharp, structured copy of the real anatomy. That makes it dangerous. A ghost copy can look exactly like a real structure, and no simple filter can tell the two apart.

Real scanners can still use regular sampling, but only together with information from multiple receiver coils (*parallel imaging*, e.g. SENSE or GRAPPA). With one coil, as here, we need a different idea.

## 4. The point spread function: predicting artifacts without an image

Zero-filled reconstruction multiplies k-space by the mask. Multiplication in k-space is convolution in the image:

$$ \text{zero-filled image} = \text{true image} \ast \text{PSF}, \qquad \text{PSF} = \mathcal{F}^{-1}\{\text{mask}\} $$

The point spread function (PSF) shows what a single bright point would turn into. Every point of the object is spread out in the same way, so the PSF of a mask predicts its artifacts before you reconstruct anything.

Because our masks select whole rows, the PSF only spreads things vertically, so a 1D plot along the vertical direction is enough.

> **Predict:** What does the PSF of a regular R = 4 mask look like? And the PSF of a mask with the same number of lines, chosen at random?

In [ ]:
def psf(lines):
    """1D point spread function of a line pattern, scaled so its central peak is 1."""
    p = np.abs(np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(lines.astype(float)))))
    return p / p.max()


offset = np.arange(N) - N // 2
patterns = {
    "regular, R = 4": regular_lines(N, 4),
    "random, R = 4": random_lines(N, 4, center_fraction=0.0),
}

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True, sharey=True)
for ax, (name, lines), color in zip(axes, patterns.items(), COLORS):
    ax.plot(offset, psf(lines), color=color, linewidth=2)
    ax.set_title(name, loc="left")
    ax.set_ylabel("PSF (peak = 1)")
    tidy(ax)
axes[-1].set_xlabel("vertical offset from the point (pixels)")
plt.tight_layout()
plt.show()

- **Regular:** the PSF has R equal spikes, $N/R = 64$ pixels apart. Every point is copied R times at full sharpness: exactly the ghost copies of Section 3.
- **Random:** one central spike, plus many small, irregular side lobes spread everywhere. Every point leaks a little energy all over the image in an irregular pattern. The artifact looks like noise instead of copies. This is called incoherent aliasing.

Same number of lines, same scan time, completely different kind of artifact.

## 5. Random undersampling

> **Predict:** Using what you learned in Notebook 1 about the center of k-space, what could go wrong if the random lines happen to miss most of the center? How could you fix it?

In [ ]:
random_no_center = random_lines(N, 4, center_fraction=0.0)
random_center = random_lines(N, 4, center_fraction=0.08)

recon_a = zero_filled(kspace, random_no_center)
recon_b = zero_filled(kspace, random_center)

show([as_mask(random_no_center), as_mask(random_center)],
     ["random lines, R = 4", "random + central 8%, R = 4"])
show([recon_a, recon_b],
     [f"NRMSE = {nrmse(recon_a, image):.3f}", f"NRMSE = {nrmse(recon_b, image):.3f}"],
     vmin=0, vmax=1)

The artifacts are now noise-like streaks along the vertical direction instead of copies. But with purely random lines, much of the center of k-space is missing, and Notebook 1 showed that the center holds most of the energy and the contrast. The image loses brightness and contrast.

Measuring a small fully sampled central block (here the central 8% of lines, about 20 lines) fixes this and lowers the error. Almost every practical sampling pattern does this.

## 6. Variable-density sampling

Notebook 1 (Exercise 3) showed that energy falls off quickly away from the center of k-space. So instead of choosing the outer lines uniformly, we can choose lines near the center more often and lines near the edges less often. This is variable-density sampling, the standard choice for compressed sensing and deep learning reconstruction. For example, the public fastMRI dataset (Zbontar et al., 2018) uses random masks with a fully sampled center: the central 8% of lines at R = 4.

Now compare all four strategies at the same acceleration, R = 4 (64 of 256 lines).

In [ ]:
R = 4
strategies = {
    "regular": regular_lines(N, R),
    "random": random_lines(N, R, center_fraction=0.0),
    "random + center": random_lines(N, R, center_fraction=0.08),
    "variable density": variable_density_lines(N, R, center_fraction=0.08),
}
recons = {name: zero_filled(kspace, lines) for name, lines in strategies.items()}

show([as_mask(lines) for lines in strategies.values()], list(strategies))
show(list(recons.values()), list(strategies), vmin=0, vmax=1)
show([np.abs(r - image) for r in recons.values()], [f"error: {name}" for name in strategies], vmin=0, vmax=0.5)

print(f"{'strategy':<18}{'lines':>6}{'NRMSE':>8}")
for name, lines in strategies.items():
    print(f"{name:<18}{lines.sum():>6}{nrmse(recons[name], image):>8.3f}")

All four patterns measure exactly 64 lines, so the scan time is identical, yet the results differ a lot.

- **Regular** gives sharp ghost copies and a high error.
- **Random** without a center loses contrast and has the highest error of all.
- **Random + center** and variable density keep the contrast and turn the remaining artifact into noise-like streaks. Variable density has the lowest error, because it spends its lines where the energy of the image is.

Look closely at the error maps: the regular pattern's error is the anatomy itself (shifted copies), while the error of the random patterns is spread out and irregular. Keep this difference in mind for the next section.

## 7. Why noise-like artifacts are good news

Here is the key idea behind the rest of this module. Real images have structure, and noise-like artifacts do not. So an algorithm that knows what real images look like can separate the two. Ghost copies, on the other hand, have exactly the same structure as the object, so no algorithm can tell which copy is real.

The clearest way to see this is with a sparse object: an image that is almost entirely zero. Below we make one with 20 bright dots, like small blood vessels in an angiogram, undersample it 4× and try the crudest possible "algorithm": keep only pixels brighter than half of the brightest pixel.

> **Predict:** With which sampling pattern will this threshold find the 20 dots without finding false ones: regular or random?

In [ ]:
from scipy.ndimage import maximum_filter

# A sparse object: 20 bright dots on an empty background
dot_rng = np.random.default_rng(1)
dots = np.zeros((N, N))
dots.flat[dot_rng.choice(N * N, 20, replace=False)] = dot_rng.uniform(0.5, 1.0, 20)
dots_kspace = fft2c(dots)

def enlarge(img):
    # display only: grow each pixel to 5 x 5 so single dots are visible on screen
    return maximum_filter(img, size=5)

panels, titles = [enlarge(dots)], ["true: 20 dots"]
for name in ["regular", "random"]:
    recon = zero_filled(dots_kspace, strategies[name])
    kept = recon > 0.5 * recon.max()
    real = (kept & (dots > 0)).sum()
    false = (kept & (dots == 0)).sum()
    print(f"{name:<8}: threshold keeps {real} real dots and {false} false ones")
    panels += [enlarge(recon), enlarge(kept)]
    titles += [f"{name}, zero-filled", f"{name}, thresholded"]

show(panels[:3], titles[:3])
show(panels[3:], titles[3:])

- **Regular:** every dot has 3 ghost copies exactly as bright as itself, so the threshold keeps 80 dots: 20 real and 60 ghosts. Nothing in the image says which ones are real.
- **Random:** each dot leaks a little energy everywhere, but the leakage is much weaker than the dot itself. The threshold keeps exactly the 20 real dots.

(For dots, uniform random sampling works best, because a single dot contains all spatial frequencies equally. Real images concentrate their energy in the center of k-space, which is why variable density won in Section 6.)

Real MR images are not sparse like this. But they are approximately sparse in other representations, for example when you look only at their edges. That is the principle compressed sensing (Notebook 3) and deep learning reconstruction (Notebook 4) build on:

1. Sample k-space so that the artifacts are incoherent (noise-like).
2. Use prior knowledge about what real images look like to separate image from artifact.
3. Keep the result consistent with the k-space lines that were actually measured.

## 8. How far can we push R?

> **Predict:** With variable-density sampling and plain zero-filled reconstruction, how does the error grow from R = 2 to R = 8? Roughly linearly, or faster?

In [ ]:
Rs = [2, 3, 4, 6, 8]
errors = [nrmse(zero_filled(kspace, variable_density_lines(N, R)), image) for R in Rs]

for R, e in zip(Rs, errors):
    print(f"R = {R}: {N // R:>3} lines, NRMSE = {e:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(Rs, errors, "o-", color=COLORS[0], linewidth=2, markersize=8)
ax.set_xlabel("acceleration factor R")
ax.set_ylabel("NRMSE (lower is better)")
ax.set_title("Zero-filled error grows with acceleration", loc="left")
ax.set_xticks(Rs)
ax.set_ylim(bottom=0)
tidy(ax)
plt.tight_layout()
plt.show()

show([zero_filled(kspace, variable_density_lines(N, R)) for R in [2, 4, 8]],
     ["R = 2", "R = 4", "R = 8"], vmin=0, vmax=1)

Zero-filled reconstruction is acceptable at R = 2 but degrades steadily: by R = 8 the small structures are lost in the artifacts. Zero-filling ignores everything we know about images. The next notebooks add that knowledge, first with mathematics (compressed sensing), then by learning it from data (deep learning).

## Exercises

Try each exercise before opening the hint. Solutions are in `solutions/02_solutions.ipynb`.

### Exercise 1: the wrong direction

Undersample columns ($k_x$, the readout direction) instead of rows: keep every 4th column. Where do the copies appear now? Why do real scanners never accelerate this way?

<details><summary>Hint</summary>

Use <code>kspace * quarter[None, :]</code> to apply a 1D pattern to columns. For the second question, reread Section 2: how long does it take to record a whole row?
</details>

In [ ]:
# Exercise 1: your code here


### Exercise 2: how big should the center be?

For variable-density sampling at R = 4, try `center_fraction` = 0, 0.02, 0.04, 0.08, 0.16 and 0.24, and plot NRMSE against it. Is a bigger center always better? Explain the trade-off.

<details><summary>Hint</summary>

The total number of lines is fixed at 64. Every line spent on the center is a line not spent on higher frequencies. Don't only look at the numbers: display the images for the smallest and the largest center, and compare the small ellipses near the bottom of the phantom.
</details>

In [ ]:
# Exercise 2: your code here


### Exercise 3: the luck of the draw

Random masks depend on the random seed. Create random + center masks (R = 4, `center_fraction=0.08`) with seeds 0 to 19 and compute the NRMSE of each. Report the mean, the best and the worst. Why should a research paper never show only its best random mask?

In [ ]:
# Exercise 3: your code here


### Exercise 4: design challenge

Using exactly 64 lines (R = 4), design the sampling pattern with the lowest zero-filled NRMSE you can. You may change `center_fraction`, `decay` and `seed`, or build a pattern completely by hand. Can you beat the variable-density pattern from Section 6? Which features did your best pattern have?

<details><summary>Hint</summary>

A loop over a few values of <code>center_fraction</code> and <code>decay</code> is a good start. Always check <code>lines.sum() == 64</code>, otherwise the comparison is unfair.
</details>

In [ ]:
# Exercise 4: your code here


## Summary

- Scanners fill k-space line by line, so skipping lines makes a scan faster. Measuring $N/R$ lines gives an acceleration factor of R.
- **Regular** undersampling gives R sharp copies of the object (coherent aliasing), which cannot be separated from real anatomy with a single coil.
- The point spread function, the inverse Fourier transform of the mask, predicts the artifact of any sampling pattern.
- **Random** sampling turns copies into noise-like (incoherent) artifacts. A fully sampled center keeps the contrast, and variable-density sampling spends lines where the energy is.
- Noise-like artifacts can be separated from structured images. This is the foundation of compressed sensing and deep learning reconstruction.

**Next:** Notebook 3, *Compressed sensing*: removing incoherent artifacts while staying consistent with the measured data.

## References

1. Lustig, M., Donoho, D., & Pauly, J. M. (2007). Sparse MRI: The application of compressed sensing for rapid MR imaging. *Magnetic Resonance in Medicine*, 58(6), 1182–1195.
2. Zbontar, J., et al. (2018). fastMRI: An open dataset and benchmarks for accelerated MRI. *arXiv:1811.08839*.
3. Nishimura, D. G. (2010). *Principles of Magnetic Resonance Imaging*. Stanford University.
4. Pruessmann, K. P., Weiger, M., Scheidegger, M. B., & Boesiger, P. (1999). SENSE: Sensitivity encoding for fast MRI. *Magnetic Resonance in Medicine*, 42(5), 952–962.

---
*Text and figures: CC-BY 4.0. Code: MIT License.*